In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import keras.ops as K
from tensorflow.keras.layers import Layer

In [3]:
CSV_PATH = "training_data_final.csv"     
MODEL_DIR = "deepfm_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# use absolute path to avoid cwd issues when running from different places
PREPROC_PATH = os.path.abspath(os.path.join(MODEL_DIR, "preproc.joblib"))
MODEL_PATH = os.path.join(MODEL_DIR, "saved_model")

EMB_DIM = 8  # embedding size

In [4]:
# 1. LOAD DATA
# -------------------------------------------------------
df = pd.read_csv(CSV_PATH)

# Ensure label column exists
df["label"] = df["label"].astype(int)

# Handle date
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [5]:
# -------------------------------------------------------
# Auto-detect numeric + categorical columns
# -------------------------------------------------------

ignore_cols = ["user_id", "label", "date"]

numeric_cols = []
categorical_cols = []

for col in df.columns:
    if col in ignore_cols:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)
    else:
        # Try converting to numeric
        try:
            pd.to_numeric(df[col])
            numeric_cols.append(col)
        except:
            categorical_cols.append(col)

# Move habit_id to categorical
if "habit_id" in numeric_cols:
    numeric_cols.remove("habit_id")
    categorical_cols.append("habit_id")

# difficulty, time_min, indoor MUST be categorical
for col in ["difficulty", "time_min", "indoor"]:
    if col in numeric_cols:
        numeric_cols.remove(col)
    if col not in categorical_cols:
        categorical_cols.append(col)

print("Detected numeric cols:", numeric_cols)
print("Detected categorical cols:", categorical_cols)


Detected numeric cols: ['risk_score', 'prediction', 'emotion_score', 'screen_time', 'unlocks', 'sleep_hours', 'steps_last_24h', 'popularity_prior']
Detected categorical cols: ['dominant_emotion', 'category', 'dopamine_level', 'recommended_for_state', 'required_device', 'habit_id', 'difficulty', 'time_min', 'indoor']


In [6]:
# -------------------------------------------------------
# Clean missing values properly
# -------------------------------------------------------

# numeric columns → convert invalid to NaN and fill with median
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

# categorical columns → convert everything to string
for c in categorical_cols:
    df[c] = df[c].astype(str).fillna("unknown")


In [7]:
# -------------------------------------------------------
# Encoding + scaling
# -------------------------------------------------------
encoders = {}

# Encode categoricals
for c in categorical_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    encoders[c] = le

# Scale numeric
scaler = None
if len(numeric_cols) > 0:
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Save preprocessing objects
preproc = {
    "encoders": encoders,
    "scaler": scaler,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols
}
joblib.dump(preproc, PREPROC_PATH)
print("Saved preprocessing to:", PREPROC_PATH)


Saved preprocessing to: x:\Documents\Project_work\Neural-habit-project\NH_fast\NeuralHabitFast\backend\Recommendation\deepfm_model\preproc.joblib


In [8]:
# 5. Prepare Inputs
# -------------------------------------------------------
X_cat = {c: df[c].values for c in categorical_cols}
X_num = df[numeric_cols].values if numeric_cols else None
y = df["label"].values

# Train-test split
train_idx, val_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

def build_inputs(idx):
    inp = {}
    for c in categorical_cols:
        inp[c] = X_cat[c][idx]
    if numeric_cols:
        inp["numeric_input"] = X_num[idx]
    return inp

train_inputs = build_inputs(train_idx)
val_inputs = build_inputs(val_idx)

y_train = y[train_idx]
y_val = y[val_idx]

# Keras expects dict: {"inp_colname": array}
def to_keras_inputs(inp_dict):
    out = {}
    for c in categorical_cols:
        out[f"inp_{c}"] = inp_dict[c].astype("int32")
    if numeric_cols:
        out["numeric_input"] = inp_dict["numeric_input"].astype("float32")
    return out

train_keras_inputs = to_keras_inputs(train_inputs)
val_keras_inputs = to_keras_inputs(val_inputs)


In [9]:
# 6. Custom FM Layer (no KerasTensor errors)
# -------------------------------------------------------
class FMLayer(Layer):
    def call(self, inputs):
        """
        inputs shape: (batch, num_fields, emb_dim)
        """
        summed = K.sum(inputs, axis=1)             # (batch, emb_dim)
        summed_square = K.square(summed)

        squared = K.square(inputs)
        squared_sum = K.sum(squared, axis=1)

        fm_output = 0.5 * K.sum(summed_square - squared_sum, axis=1, keepdims=True)
        return fm_output


In [10]:
# 7. Build DeepFM model
# -------------------------------------------------------
inputs = {}
emb_fm_list = []     # (batch, 1, emb_dim)
emb_dnn_list = []    # (batch, emb_dim)

for c in categorical_cols:
    vocab_size = int(df[c].nunique()) + 1
    inp = Input(shape=(1,), name=f"inp_{c}")
    inputs[c] = inp

    emb = layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMB_DIM,
        name=f"emb_{c}"
    )(inp)

    emb_fm_list.append(emb)                      # shape (batch,1,emb_dim)
    emb_dnn_list.append(layers.Reshape((EMB_DIM,))(emb))

# Numeric input
if numeric_cols:
    num_inp = Input(shape=(len(numeric_cols),), name="numeric_input")
    inputs["numeric_input"] = num_inp

    # numeric as FM field
    num_fm = layers.Dense(EMB_DIM)(num_inp)
    num_fm = layers.Reshape((1, EMB_DIM))(num_fm)
    emb_fm_list.append(num_fm)

    # numeric passed directly to DNN
    emb_dnn_list.append(num_inp)

# -------- FM Component --------
fm_tensor = layers.Concatenate(axis=1)(emb_fm_list)
fm_output = FMLayer()(fm_tensor)

# -------- Linear Component (very simple) --------
linear_input = layers.Concatenate()(emb_dnn_list)
linear_logit = layers.Dense(1)(linear_input)

# -------- Deep Component (MLP) --------
dnn_input = layers.Concatenate()(emb_dnn_list)
x = layers.Dense(128, activation="relu")(dnn_input)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
dnn_logit = layers.Dense(1)(x)

# -------- Final Output --------
final_logit = layers.Add()([fm_output, linear_logit, dnn_logit])
output = layers.Activation("sigmoid")(final_logit)

model = Model(inputs=list(inputs.values()), outputs=output)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_dominant_emoti… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_category        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_dopamine_level  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_recommended_fo… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_required_device │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_habit_id        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_difficulty      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_time_min        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_indoor          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dominant_emoti… │ (None, 1, 8)      │        232 │ inp_dominant_emo… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_category        │ (None, 1, 8)      │         32 │ inp_category[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dopamine_level  │ (None, 1, 8)      │         32 │ inp_dopamine_lev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_recommended_fo… │ (None, 1, 8)      │      1,112 │ inp_recommended_… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_required_device │ (None, 1, 8)      │         32 │ inp_required_dev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_habit_id        │ (None, 1, 8)      │      2,408 │ inp_habit_id[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_difficulty      │ (None, 1, 8)      │         48 │ inp_difficulty[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_time_min        │ (None, 1, 8)      │         96 │ inp_time_min[0][

 Total params: 22,858 (89.29 KB)

 Trainable params: 22,858 (89.29 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 8. Train model
# -------------------------------------------------------
callbacks = [
    EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True),
    ModelCheckpoint(os.path.join(MODEL_DIR, "best_model.keras"), save_best_only=True, monitor="val_auc", mode="max")
]

history = model.fit(
    train_keras_inputs, y_train,
    validation_data=(val_keras_inputs, y_val),
    epochs=25,
    batch_size=512,
    callbacks=callbacks,
    verbose=2
)


Epoch 1/25
32/32 - 4s - 120ms/step - auc: 0.6717 - loss: 0.6036 - val_auc: 0.7285 - val_loss: 0.5696
Epoch 2/25
32/32 - 0s - 8ms/step - auc: 0.7366 - loss: 0.5608 - val_auc: 0.7385 - val_loss: 0.5607
Epoch 3/25
32/32 - 1s - 46ms/step - auc: 0.7494 - loss: 0.5505 - val_auc: 0.7441 - val_loss: 0.5571
Epoch 4/25
32/32 - 0s - 8ms/step - auc: 0.7588 - loss: 0.5435 - val_auc: 0.7475 - val_loss: 0.5547
Epoch 5/25
32/32 - 0s - 8ms/step - auc: 0.7675 - loss: 0.5360 - val_auc: 0.7516 - val_loss: 0.5528
Epoch 6/25
32/32 - 0s - 9ms/step - auc: 0.7727 - loss: 0.5317 - val_auc: 0.7534 - val_loss: 0.5520
Epoch 7/25
32/32 - 0s - 8ms/step - auc: 0.7798 - loss: 0.5264 - val_auc: 0.7547 - val_loss: 0.5514
Epoch 8/25
32/32 - 0s - 6ms/step - auc: 0.7848 - loss: 0.5217 - val_auc: 0.7540 - val_loss: 0.5532
Epoch 9/25
32/32 - 0s - 5ms/step - auc: 0.7883 - loss: 0.5183 - val_auc: 0.7539 - val_loss: 0.5542
Epoch 10/25
32/32 - 0s - 6ms/step - auc: 0.7919 - loss: 0.5145 - val_auc: 0.7515 - val_loss: 0.5568
Epoch 

In [12]:
# 9. Evaluate
# -------------------------------------------------------
val_pred = model.predict(val_keras_inputs).ravel()
val_bin = (val_pred >= 0.5).astype(int)

print("\n----- Validation Report -----")
print(classification_report(y_val, val_bin))
print("ROC AUC:", roc_auc_score(y_val, val_pred))


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

----- Validation Report -----
              precision    recall  f1-score   support

           0       0.63      0.41      0.50      1396
           1       0.73      0.87      0.80      2604

    accuracy                           0.71      4000
   macro avg       0.68      0.64      0.65      4000
weighted avg       0.70      0.71      0.69      4000

ROC AUC: 0.7548091100753085


In [13]:
# -------------------------------------------------------
# 10. Save final model (Keras 3 compatible)
# -------------------------------------------------------

# Save as .keras file (recommended)
MODEL_PATH = os.path.join(MODEL_DIR, "saved_model.keras")

model.save(MODEL_PATH)
print("\nModel saved to:", MODEL_PATH)

# Preprocessing already saved earlier
print("Preprocessing saved to:", PREPROC_PATH)



Model saved to: deepfm_model\saved_model.keras
Preprocessing saved to: x:\Documents\Project_work\Neural-habit-project\NH_fast\NeuralHabitFast\backend\Recommendation\deepfm_model\preproc.joblib


In [14]:
# 11. Inference + Recommendation Function
# -------------------------------------------------------
def load_model_and_preproc():
    pre = joblib.load(PREPROC_PATH)
    mdl = tf.keras.models.load_model(MODEL_PATH, custom_objects={"FMLayer": FMLayer})
    return pre, mdl

def normalize_recommended_state(val):
    # training likely stored scalar strings. Handle lists robustly.
    # if pd.isna(val):
    #     return "unknown"
    if isinstance(val, list):
        return val[0] if len(val) > 0 else "unknown"   # PICK-FIRST heuristic
    if isinstance(val, (set, tuple)):
        return list(val)[0] if len(val) > 0 else "unknown"
    if isinstance(val, dict):
        return "unknown"
    # if it's already a string like "stress|anxious", keep it
    return str(val)

def build_candidate_table_from_user(user_row: dict, habit_catalog: pd.DataFrame):
    """
    user_row: dict or single-row DataFrame containing user features:
      risk_score,prediction,dominant_emotion,emotion_score,screen_time,unlocks,
      sleep_hours,steps_last_24h (names must match preproc)
    habit_catalog: DataFrame with habit columns:
      habit_id,category,difficulty,time_min,dopamine_level,recommended_for_state,
      indoor,required_device,popularity_prior
    returns: df where user features are repeated for each habit
    """
    # convert user_row to DataFrame (1,row) if needed
    if isinstance(user_row, dict):
        df_user = pd.DataFrame([user_row])
    else:
        df_user = user_row.copy().reset_index(drop=True)
    # normalize recommended_for_state in habit catalog and user if present
    if "recommended_for_state" in habit_catalog.columns:
        habit_catalog = habit_catalog.copy()
        habit_catalog["recommended_for_state"] = habit_catalog["recommended_for_state"].apply(normalize_recommended_state)
    # repeat user row for each habit
    n = len(habit_catalog)
    df_user_rep = pd.concat([df_user]*n, ignore_index=True)
    # concat user features and habit features side-by-side
    # ensure no column name collision: if both have same col (like 'date'/'user_id'), handle explicitly
    df = pd.concat([df_user_rep.reset_index(drop=True), habit_catalog.reset_index(drop=True)], axis=1)
    return df

def preprocess_candidates(df, pre):
    encs = pre["encoders"]
    scaler = pre["scaler"]
    cat_cols = pre["categorical_cols"]
    num_cols = pre["numeric_cols"]

    df = df.copy()

    # categorical
    for c in cat_cols:
        le = encs[c]
        df[c] = df[c].astype(str).fillna("unknown")
        # map using label encoder, unknown → first class
        df[c] = df[c].apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else 0
        )

    # numeric
    if num_cols:
        for c in num_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
        df[num_cols] = scaler.transform(df[num_cols])

    # convert to model inputs
    inp = {}
    for c in cat_cols:
        inp[f"inp_{c}"] = df[c].values.astype("int32")
    if num_cols:
        inp["numeric_input"] = df[num_cols].values.astype("float32")

    return inp, df


def recommend_for_user_snapshot(user_row, habit_catalog, top_positive=5, top_negative=5):
    """
    user_row: dict or single-row DataFrame with user features (single snapshot)
    habit_catalog: DataFrame with habit rows
    returns: pos_df, neg_df containing habit_id and score
    """
    pre, mdl = load_model_and_preproc()

    # build candidate table
    df_cand = build_candidate_table_from_user(user_row, habit_catalog)

    # preprocess whole candidate set
    inp, df_proc = preprocess_candidates(df_cand, pre)

    # predict scores
    scores = mdl.predict(inp, batch_size=1024).ravel()
    df_proc["score"] = scores

    # sort & return top/bottom
    pos = df_proc.sort_values("score", ascending=False).head(top_positive)[["habit_id", "score"]].reset_index(drop=True)
    neg = df_proc.sort_values("score", ascending=True).head(top_negative)[["habit_id", "score"]].reset_index(drop=True)
    return pos, neg


print("\nDeepFM training complete.")


DeepFM training complete.


In [26]:
# user snapshot as dict (names must match preproc)
user_snapshot = {
    "risk_score": 0.22,
    "prediction": 0,
    "dominant_emotion": "Pride",
    "emotion_score": -0.76,
    "screen_time": 247,
    "unlocks": 26,
    "sleep_hours": 6.1,
    "steps_last_24h": 2870
}

# habit_catalog: load your CSV or JSON into df_habits
df_habits = pd.read_json("habit_catalog.json")   # or your DataFrame

pos5, neg5 = recommend_for_user_snapshot(user_snapshot, df_habits, top_positive=5, top_negative=5)
print("Top 5 recs:\n", pos5)
print("Least 5:\n", neg5)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
Top 5 recs:
    habit_id     score
0        18  0.972298
1        95  0.969227
2       251  0.966268
3       149  0.962101
4       267  0.961893
Least 5:
    habit_id     score
0       205  0.057434
1       298  0.126162
2        87  0.145337
3       223  0.158528
4       170  0.170192
